In [ ]:
import duckdb
import os
from deltalake.writer import write_deltalake

delta_table_1 = "path/to/delta1"
delta_table_2 = "path/to/delta2"
output_delta_table = "delta_output"

batch_size = 3_000_000

## Fazendo um count antes

In [ ]:
con = duckdb.connect()

query = f"""
    SELECT COUNT(*) as total_rows
    FROM delta_scan({delta_table_1})
    """

df = con.execute(query).pl()
df

query = f"""
    SELECT COUNT(*) as total_rows
    FROM delta_scan({delta_table_2})
    """

df = con.execute(query).pl()
df

In [ ]:
import duckdb
from deltalake.writer import write_deltalake

# Connect to DuckDB
con = duckdb.connect()

# Define paths to the Delta tables
path_deltalake1 = "path/to/delta1"
path_deltalake2 = "path/to/delta2"
output_path = "delta_output"

# Total rows per batch
batch_size = 3_000_000

# Step 1: Count total rows (optional for verification)
query_total_1 = f"SELECT COUNT(*) as total_rows FROM delta_scan('{path_deltalake1}')"
query_total_2 = f"SELECT COUNT(*) as total_rows FROM delta_scan('{path_deltalake2}')"

total_rows_1 = con.execute(query_total_1).fetchone()[0]
total_rows_2 = con.execute(query_total_2).fetchone()[0]

con.close()

print(f"Total Rows in DeltaLake1: {total_rows_1}")
print(f"Total Rows in DeltaLake2: {total_rows_2}")

## Fazendo join e excluindo valores duplicados.
- Essa função verifica onde o filename de t2 é igual ao de t1. 
- Se tem no 2 mas não tem no 1, o do 1 ficará nulo.
- Portanto será mantido somente os que tem no 2 mas não tem no 1.

In [ ]:
con = duckdb.connect()
# Step 2: Perform the join to remove duplicates
query_join = f"""
    SELECT t2.path, t2.filename
    FROM delta_scan('{path_deltalake2}') t2
    LEFT JOIN delta_scan('{path_deltalake1}') t1
        ON t2.filename = t1.filename
    WHERE t1.filename IS NULL
"""

# Step 3: Export data in batches
offset = 0

In [ ]:
while True:
    batch_query = f"""
        {query_join}
        LIMIT {batch_size}
        OFFSET {offset}
    """
    # Fetch the batch as a Pandas DataFrame
    batch_df = con.execute(batch_query).df()

    if batch_df.empty:
        print("No more rows to process.")
        break

    # Append the batch to the new Delta table
    write_deltalake(output_path, batch_df, mode="append")
    
    print(f"Processed batch with offset {offset}, size {len(batch_df)}")
    offset += batch_size

print("Processing completed.")